# Idea

Tabled Asymmetric Numeral Systems (tANS) is an entropy coding scheme. It allows compressing data optimally close to Shannon entropy, just like Arithmetic Coding, but at the speed of Huffman coding due to a precomputed lookup table.

For our task of packing string identifiers into bit vectors:
1. We compute the frequency of characters (a-z, 0-9, and separator) based on an English dictionary (modelling realistic identifiers).
2. We assign probabilities to each character. Frequent characters (like 'e', 'a', separator) take fractional bits < 5, while rare characters ('z', 'q') take more bits.
3. Because tANS operates as a Finite State Machine (FSM), encoding and decoding is an $O(1)$ table lookup sequence without big integer arithmetic, unlike base encodings.

# Density

In [3]:
import urllib.request
import collections
import math

# Download english words to build a realistic dataset
url = "https://raw.githubusercontent.com/dwyl/english-words/master/words_alpha.txt"
try:
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    response = urllib.request.urlopen(req)
    words = response.read().decode('utf-8').splitlines()
except Exception as e:
    # Small fallback dictionary if network fails
    words = ["example", "test", "identifier", "count", "string", "vector", "player", "manager"]

print(f"Loaded {len(words)} words.")

# Count character frequencies
counter = collections.Counter()
for word in words:
    counter.update(word.lower())

# Keep only a-z
letters_count = {chr(i): counter.get(chr(i), 1) for i in range(97, 123)}
total_letters = sum(letters_count.values())

# Model identifier symbols (letters + digits + separator)
# Assuming 1 separator per token, and average token length is average word length
avg_word_length = total_letters / len(words)
separator_count = int(total_letters / avg_word_length)

# Assuming digits act as maybe 5% of all characters
# digits_total / (total_letters + separator_count + digits_total) = 0.05
# digits_total = 0.05 / 0.95 * (total_letters + separator_count)
digits_total = int((total_letters + separator_count) * 0.05 / 0.95)
digit_count = digits_total // 10  # evenly distributed

freqs = letters_count.copy()
for i in range(10):
    freqs[str(i)] = digit_count
freqs['_'] = separator_count    # separator

total_freq = sum(freqs.values())
probs = {k: v / total_freq for k, v in freqs.items()}

# Theoretical bits per symbol based on Shannon Entropy (-log2(P))
bits_per_symbol = {k: -math.log2(p) for k, p in probs.items()}

best_char = min(bits_per_symbol, key=bits_per_symbol.get)
worst_char = max(bits_per_symbol, key=bits_per_symbol.get)

H = sum(p * bits_per_symbol[k] for k, p in probs.items())
print(f"Average token length from dictionary: {avg_word_length:.2f} letters")
print(f"Shannon Entropy (Avg bits/symbol): {H:.3f}")
print(f"Best char: '{best_char}' -> {bits_per_symbol[best_char]:.3f} bits")
print(f"Worst char: '{worst_char}' -> {bits_per_symbol[worst_char]:.3f} bits")

Loaded 370105 words.
Average token length from dictionary: 9.44 letters
Shannon Entropy (Avg bits/symbol): 4.492
Best char: 'e' -> 3.434 bits
Worst char: 'j' -> 9.542 bits


In [4]:
vector_lengths = [64, 128, 192, 256]

for vl in vector_lengths:
    avg_symbols = vl / H
    best_symbols = vl / bits_per_symbol[best_char]
    worst_symbols = vl / bits_per_symbol[worst_char]
    print(f"{vl}-bits vector constraints:")
    print(f"  Avg capacity: ~{avg_symbols:.1f} symbols (Density: {H:.3f} bit/sym)")
    print(f"  Best case ('{best_char}'): ~{best_symbols:.1f} symbols (Density: {bits_per_symbol[best_char]:.3f} bit/sym)")
    print(f"  Worst case ('{worst_char}'): ~{worst_symbols:.1f} symbols (Density: {bits_per_symbol[worst_char]:.3f} bit/sym)\n")

64-bits vector constraints:
  Avg capacity: ~14.2 symbols (Density: 4.492 bit/sym)
  Best case ('e'): ~18.6 symbols (Density: 3.434 bit/sym)
  Worst case ('j'): ~6.7 symbols (Density: 9.542 bit/sym)

128-bits vector constraints:
  Avg capacity: ~28.5 symbols (Density: 4.492 bit/sym)
  Best case ('e'): ~37.3 symbols (Density: 3.434 bit/sym)
  Worst case ('j'): ~13.4 symbols (Density: 9.542 bit/sym)

192-bits vector constraints:
  Avg capacity: ~42.7 symbols (Density: 4.492 bit/sym)
  Best case ('e'): ~55.9 symbols (Density: 3.434 bit/sym)
  Worst case ('j'): ~20.1 symbols (Density: 9.542 bit/sym)

256-bits vector constraints:
  Avg capacity: ~57.0 symbols (Density: 4.492 bit/sym)
  Best case ('e'): ~74.6 symbols (Density: 3.434 bit/sym)
  Worst case ('j'): ~26.8 symbols (Density: 9.542 bit/sym)



# Operations

Unlike bases like 32 or 64 which allow fast random access, tANS encoding processes the string statefully. Because the whole identifier is packed as a single combined state:
- `Encode` ~ $O(L_{id})$ but extremely fast (table lookup: `state = encode_table[state][symbol]`). No expensive large division operations.
- `Decode` ~ $O(L_{id})$ with table lookup (`symbol = decode_table[state].symbol; state = decode_table[state].next`). Extremely fast.
- `GetLength` ~ $O(L_{id})$ - requires full decode to count (unless length is stored explicitly).
- `GetSymbol` / `GetToken` ~ $O(L_{id})$ - Requires sequential decoding from the initial state.
- `Append` / `Replace` / `Insert` ~ $O(L_{id})$ - Any modification requires repacking at least the suffix of the string.

**Comparison vs Enumeration-EMN:**
Enumeration-EMN achieved 4.536 bits/symbol constant, but decoding is purely $O(L_{id})$ with slow BigInt division operations. 

tANS achieves **~4.492 bits/symbol** (average entropy) when assuming exactly 1 separator per real dictionary word, meaning it packs MORE symbols on average, and its encode/decode FSM operates with lightning fast memory FSM operations ($O(1)$ per symbol) instead of mathematical calculation of long division. We also get the advantage that very frequent tokens like the letter `e` encode with ~3.4 bits, which boosts the overall capacity for well-composed identifiers.
</VSCode.Cell>